# EasyRAG Code Map And Runtime Flow

## Chapter position

This chapter steps back from individual labs and reconnects the teaching path to the codebase shape, backend contracts, and observability surfaces behind it.

## Learning question

Which modules own each stage of the runtime flow, and how do those modules connect back to the public API?

## Success criteria

- trace which modules own the main RAG stage transitions
- inspect the public method signatures that connect those modules
- compare the conceptual learning path with the actual runtime entry points

## Source code anchors

- `easyrag.rag.orchestrator`
- `easyrag.rag.types`
- `easyrag.rag.indexing.pipeline`
- `easyrag.rag.retrieval.pipeline`
- `easyrag.rag.generation.pipeline`
- `docs/engineering/20-code-map.md`


## Method principles

- `easyrag.rag.orchestrator`: This module is the public lifecycle hub. It gathers the stage-specific packages behind a single orchestrator class and a small number of entry points.
- `easyrag.rag.types`: This module defines the shared data contracts that move across stages. It keeps retrieval, generation, and evaluation aligned on the same object shapes.
- `easyrag.rag.indexing.pipeline`: This module owns the fan-out from canonical documents to storage payloads and persisted indexing artifacts. It is where indexing becomes concrete.
- `easyrag.rag.retrieval.pipeline`: This module owns end-to-end query execution. It is the place where preparation, recall, filtering, hydration, and result assembly are stitched together.
- `easyrag.rag.generation.pipeline`: This module owns answer orchestration on top of `QueryResult`. It keeps citation selection, packing, prompting, and synthesis in one readable flow.
- `docs/engineering/20-code-map.md`: This engineering note is the human-oriented map back to the codebase. It connects the tutorial stages to the source files that actually own them.


## How the code fits together

The flow in this notebook is module files -> public signatures -> stage ownership map. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

## What this cell is proving

We start by making the repo importable from the notebook runtime and loading the helpers this notebook depends on. This cell should stay quiet. It is only here to make the later examples reliable.

In [ ]:
# ruff: noqa: F401
import importlib
import inspect
import re
import sys
import tempfile
from pathlib import Path
from pprint import pprint

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_rag = importlib.import_module("easyrag.rag")
_providers = importlib.import_module("easyrag.rag.providers")
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
can_use_openai_compatible_models = _providers.can_use_openai_compatible_models

REPO_ROOT = Path.cwd()
module_map = {
    "Public orchestrator": REPO_ROOT / "easyrag/rag/orchestrator.py",
    "Shared types": REPO_ROOT / "easyrag/rag/types.py",
    "Indexing pipeline": REPO_ROOT / "easyrag/rag/indexing/pipeline.py",
    "Retrieval pipeline": REPO_ROOT / "easyrag/rag/retrieval/pipeline.py",
    "Generation pipeline": REPO_ROOT / "easyrag/rag/generation/pipeline.py",
    "Engineering code map": REPO_ROOT / "docs/engineering/20-code-map.md",
}

## What this cell is proving

We begin by reading the files as files. That keeps the notebook honest about where each stage transition lives.

In [ ]:
for label, path in module_map.items():
    text = path.read_text()
    print(f"=== {label}: {path.relative_to(REPO_ROOT)} ===")
    if path.suffix == ".py":
        names = re.findall(
            r"^(?:async\s+def|def|class)\s+([A-Za-z0-9_]+)", text, flags=re.MULTILINE
        )
        pprint(names[:12])
    else:
        print("\n".join(text.strip().splitlines()[:12]))
    print()

## Why this output looks like this

The next cell focuses on the public surface. These signatures are the shortest path from the notebook story to the actual runtime story.

In [ ]:
signatures = {
    "EasyRAG.__init__": str(inspect.signature(EasyRAG.__init__)),
    "EasyRAG.ainsert": str(inspect.signature(EasyRAG.ainsert)),
    "EasyRAG.aquery": str(inspect.signature(EasyRAG.aquery)),
    "EasyRAG.aanswer": str(inspect.signature(EasyRAG.aanswer)),
    "EasyRAG.get_stats": str(inspect.signature(EasyRAG.get_stats)),
}
pprint(signatures)

stage_ownership = {
    "loading_and_prepare": [
        "easyrag.rag.orchestrator.EasyRAG.prepare_documents",
        "easyrag.rag.indexing.prepare.prepare_documents_for_insert",
    ],
    "indexing": [
        "easyrag.rag.indexing.chunking_core",
        "easyrag.rag.indexing.pipeline.build_insert_payloads",
        "easyrag.rag.indexing.pipeline.ingest_documents",
    ],
    "retrieval": [
        "easyrag.rag.retrieval.preprocess.QueryPreprocessor.prepare",
        "easyrag.rag.retrieval.pipeline.execute_query",
    ],
    "generation": [
        "easyrag.rag.generation.selection.select_answer_citations",
        "easyrag.rag.generation.packing.build_context_block",
        "easyrag.rag.generation.pipeline.generate_answer",
    ],
    "evaluation": [
        "easyrag.rag.evaluation.runner.evaluate_retrieval",
        "easyrag.rag.evaluation.runner.evaluate_answers",
    ],
}
pprint(stage_ownership)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

When provider-backed defaults are available, `EasyRAG` wires those callables into the orchestrator at construction time. The next cell prints that wiring instead of running a full end-to-end query.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    try:
        provider_rag = EasyRAG(
            working_dir=provider_tmp.name, workspace="provider-code-map-demo"
        )
        wiring = {
            "llm_model_func": getattr(
                provider_rag.llm_model_func,
                "__name__",
                type(provider_rag.llm_model_func).__name__,
            )
            if provider_rag.llm_model_func
            else None,
            "query_model_func": getattr(
                provider_rag.query_model_func,
                "__name__",
                type(provider_rag.query_model_func).__name__,
            )
            if provider_rag.query_model_func
            else None,
            "embedding_func": getattr(
                provider_rag.embedding_func,
                "__name__",
                type(provider_rag.embedding_func).__name__,
            )
            if provider_rag.embedding_func
            else None,
            "reranker_func": getattr(
                provider_rag.reranker_func,
                "__name__",
                type(provider_rag.reranker_func).__name__,
            )
            if provider_rag.reranker_func
            else None,
            "answer_model_func": getattr(
                provider_rag.answer_model_func,
                "__name__",
                type(provider_rag.answer_model_func).__name__,
            )
            if provider_rag.answer_model_func
            else None,
        }
        pprint(wiring)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        provider_tmp.cleanup()

## Common mistakes / debugging cues

- A stable public API does not mean the internal ownership is fuzzy. Keep module boundaries visible.
- Backend swaps should change implementation classes, not the lifecycle shape you teach or debug.
- Observability fields matter because degraded runs often still look superficially successful.

## Takeaway

The conceptual stage order from the docs still holds, but now you can point at the owning modules. `orchestrator.py` owns the public lifecycle. `types.py` defines the data contracts. The indexing, retrieval, generation, and evaluation packages own the stage-specific logic behind those public methods.

## Where to go next

- Continue with [08_02_local_vs_production_backends.ipynb](08_02_local_vs_production_backends.ipynb) if you want to compare runtime backends after this code-oriented walkthrough.
- Read [08-system-architecture-overview.md](../../docs/08-system-architecture-overview.md) for the chapter-level architecture map.
- Keep [20-code-map.md](../../docs/engineering/20-code-map.md) nearby as the lightweight engineering reference.